# LiteLLM 은 캐시 write/read 를 어떤 필드로 주는가 — 4모델 × 3턴

**LiteLLM 버전**: `1.83.7` (`proj-in4u-bxg2.0-api` 가 핀한 버전과 동일)

## 왜 이걸 확인하나

`proj-in4u-bxg2.0-api` 의 `LiteLLMProvider` 는 응답 usage 에서
`prompt_tokens` / `completion_tokens` / `reasoning_tokens` **세 개만** 읽습니다
(`litellm_provider.py` 의 `_response_to_chunk`, `_extract_reasoning_tokens`).
**캐시 토큰은 전혀 보지 않습니다.**

캐시 읽기는 입력가의 10% 로 과금되므로, 이걸 계측하지 않으면 **원가가 실제보다 부풀려 기록**됩니다.
반대로 GPT 5.6 세대는 캐시 쓰기에 1.25배가 붙어서 **과소 기록**되기도 합니다.

그래서 계측을 추가하려면 먼저 알아야 합니다 —
**LiteLLM 이 프로바이더별로 캐시 정보를 어느 필드에 담아주는가?**

## LiteLLM 은 usage 를 OpenAI 형식으로 "정규화"하지만, 완전하지 않습니다

미리 요약하면 이렇습니다 (아래 셀에서 실측으로 확인합니다).

| | 읽기(read) | 쓰기(write) |
|---|---|---|
| `openai/gpt-5.6-*` | `prompt_tokens_details.cached_tokens` | `prompt_tokens_details.cache_write_tokens` |
| `openai/gpt-5.5 이하` | `prompt_tokens_details.cached_tokens` | **필드 자체가 없음** |
| `anthropic/*` | `prompt_tokens_details.cached_tokens`<br>+ 최상위 `cache_read_input_tokens` | `prompt_tokens_details.cache_creation_tokens`<br>+ 최상위 `cache_creation_input_tokens` |
| `gemini/*` | `prompt_tokens_details.cached_tokens`<br>+ 최상위 `cache_read_input_tokens`<br>(**미스 시 둘 다 `None`**) | 없음 (쓰기 개념 없음) |

**읽기는 `cached_tokens` 하나로 통일되지만, 쓰기는 프로바이더마다 이름이 다릅니다.**
게다가 값이 없을 때 **`0` / `None` / 키 자체 부재**가 섞여 있어 방어 코드가 필요합니다.

## 실험 설계
- 각 모델을 **3턴 멀티턴**으로 호출 (턴마다 대화가 누적 → 이전 턴이 다음 턴의 프리픽스가 됨)
- 매 턴 `response.usage` 를 **원본 그대로 덤프** — 정규화하지 않고 무엇이 오는지 봅니다
- Anthropic 만 `cache_control` 을 명시해야 캐시가 걸리므로, 매 턴 마지막 user 블록으로 이동시킵니다

> ⚠️ **커널**: 이 노트북은 `litellm==1.83.7` 전용 venv 를 씁니다.
> 리서치 레포 본 환경은 `openai>=2.45` 인데 litellm 1.83.7 은 `openai==2.30.0` 을 핀해서 충돌하기 때문입니다.
> 커널을 **"Python (litellm 1.83.7)"** 으로 선택하세요. CLI 실행은 README 참고.


In [1]:
# === 셋업 ===
import json, os, pathlib, time
from importlib import metadata

from dotenv import load_dotenv

load_dotenv()  # 리서치 레포 루트 .env (OPENAI_API_KEY / ANTHROPIC_API_KEY / GOOGLE_API_KEY)

# LiteLLM 의 gemini/ 경로는 GEMINI_API_KEY 를 봅니다 — .env 에는 GOOGLE_API_KEY 로 들어있어 매핑해 줍니다.
if not os.environ.get("GEMINI_API_KEY") and os.environ.get("GOOGLE_API_KEY"):
    os.environ["GEMINI_API_KEY"] = os.environ["GOOGLE_API_KEY"]

import litellm

litellm.suppress_debug_info = True

print("litellm :", metadata.version("litellm"))
print("openai  :", metadata.version("openai"), "(litellm 1.83.7 이 핀한 버전)")
for k in ("OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GEMINI_API_KEY"):
    print(f"{k:18s}: {'있음' if os.environ.get(k) else '없음 ⚠️'}")

N_TURNS = 3

litellm : 1.83.7
openai  : 2.30.0 (litellm 1.83.7 이 핀한 버전)
OPENAI_API_KEY    : 있음
ANTHROPIC_API_KEY : 있음
GEMINI_API_KEY    : 있음


In [2]:
# === 공통 시나리오 ===
# 3사 최소 캐싱 임계(1024 토큰)를 넉넉히 넘기도록 시스템 프롬프트를 크게 만듭니다.
# Gemini 암시적 캐싱은 임계 근처에서 잘 안 걸려서 여유 있게 잡습니다.
_RULE = (
    "Operational rule: every deployment must be preceded by a configuration snapshot so that "
    "before/after comparison is possible. Staging must reproduce the change and pass regression "
    "tests first. Production changes follow the four-eyes principle and are applied only inside a "
    "scheduled maintenance window. After rollout, metrics are observed for 30 minutes and rolled "
    "back immediately on anomaly. P1 incidents require initial response within 15 minutes and an "
    "RCA report within 3 business days. "
)
SYSTEM_PROMPT = (
    "You are a site reliability engineer assistant for the 'orderhub' service. "
    "Answer strictly from the operations runbook below. Keep every answer to one short sentence.\n\n"
    + _RULE * 60
)

_LOG = (
    "[12:{t:02d}:{i:02d}] svc=order-api level=warn msg=\"retry scheduled\" attempt={n} "
    "latency_ms={lat} consumer=inbox-worker partition={p} idempotency_key=ord-{n:06d} "
)

QUESTIONS = [
    "이 로그에서 재시도가 반복되는 원인으로 가장 먼저 의심할 것은?",
    "P1 장애 초동 대응 시간과 RCA 제출 기한은?",
    "지금까지 대화를 한 문장으로 요약해줘.",
]


def user_text(turn: int) -> str:
    """턴마다 로그 발췌가 붙어 컨텍스트가 누적됩니다 (에이전트 대화 모사)."""
    log = "".join(
        _LOG.format(t=turn, i=i, n=turn * 100 + i, lat=120 + i * 7, p=i % 4) for i in range(14)
    )
    return f"[로그 발췌 turn {turn}]\n{log}\n\n질문: {QUESTIONS[turn - 1]}"


print(f"시스템 프롬프트: {len(SYSTEM_PROMPT):,}자 | 턴당 사용자 메시지: 약 {len(user_text(1)):,}자")

시스템 프롬프트: 29,627자 | 턴당 사용자 메시지: 약 2,100자


In [3]:
# === 대상 모델 ===
# needs_cache_control=True 인 모델만 cache_control 을 명시해야 캐시가 걸립니다(Anthropic).
# 나머지는 자동(OpenAI 프리픽스 캐싱 / Gemini 암시적 캐싱)이라 아무것도 안 해도 됩니다.
MODELS = [
    {"key": "openai/gpt-5.6-luna",       "label": "GPT 5.6 luna",   "needs_cache_control": False},
    {"key": "openai/gpt-5.4-mini",       "label": "GPT 5.4 mini",   "needs_cache_control": False},
    {"key": "anthropic/claude-opus-4-8", "label": "Claude Opus 4.8", "needs_cache_control": True},
    {"key": "gemini/gemini-3.6-flash",   "label": "Gemini 3.6 flash", "needs_cache_control": False},
]

for m in MODELS:
    cc = "cache_control 명시" if m["needs_cache_control"] else "자동"
    print(f"  {m['key']:32s} {cc}")

  openai/gpt-5.6-luna              자동
  openai/gpt-5.4-mini              자동
  anthropic/claude-opus-4-8        cache_control 명시
  gemini/gemini-3.6-flash          자동


In [4]:
# === 러너: 3턴 멀티턴 + usage 원본 보존 ===
def usage_to_dict(usage) -> dict:
    """usage 객체를 정규화 없이 통째로 dict 로 — 무엇이 오는지 그대로 보기 위함."""
    if usage is None:
        return {}
    if hasattr(usage, "model_dump"):
        return usage.model_dump()
    try:
        return dict(usage)
    except Exception:
        return {"repr": repr(usage)}


def run_model(cfg) -> list[dict]:
    """한 모델을 N_TURNS 턴 돌리고 턴별 usage 원본을 모아 반환."""
    model = cfg["key"]
    system_block = {"type": "text", "text": SYSTEM_PROMPT}
    if cfg["needs_cache_control"]:
        system_block["cache_control"] = {"type": "ephemeral"}
        messages = [{"role": "system", "content": [system_block]}]
    else:
        messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    rows, prev_block = [], None
    for turn in range(1, N_TURNS + 1):
        if cfg["needs_cache_control"]:
            # Anthropic: 매 턴 마지막 user 블록으로 cache_control 을 이동시켜야
            # 이전 대화까지 캐시가 확장됩니다. breakpoint 는 최대 4개라 직전 것은 회수합니다.
            if prev_block is not None:
                prev_block.pop("cache_control", None)
            block = {"type": "text", "text": user_text(turn), "cache_control": {"type": "ephemeral"}}
            messages.append({"role": "user", "content": [block]})
            prev_block = block
        else:
            messages.append({"role": "user", "content": user_text(turn)})

        t0 = time.perf_counter()
        try:
            resp = litellm.completion(model=model, messages=messages, max_tokens=64)
        except Exception as e:
            print(f"  ⚠️ 턴{turn} 실패 — 여기까지 결과로 진행: {type(e).__name__}: {str(e)[:160]}")
            break
        elapsed = time.perf_counter() - t0

        u = usage_to_dict(resp.usage)
        answer = resp.choices[0].message.content or "(빈 응답)"
        messages.append({"role": "assistant", "content": answer})
        rows.append({"turn": turn, "elapsed": elapsed, "usage": u})
        print(f"  턴{turn}: prompt={u.get('prompt_tokens')} completion={u.get('completion_tokens')} "
              f"({elapsed:.2f}s)")
    return rows

In [5]:
# === 4모델 실행 ===
results = {}
for cfg in MODELS:
    print(f"\n=== {cfg['label']} ({cfg['key']}) — {N_TURNS}턴 ===")
    results[cfg["key"]] = {"cfg": cfg, "rows": run_model(cfg)}

ok = {k: v for k, v in results.items() if v["rows"]}
print("\n완료:", ", ".join(ok) or "(없음)")


=== GPT 5.6 luna (openai/gpt-5.6-luna) — 3턴 ===


  턴1: prompt=5943 completion=64 (1.74s)


  턴2: prompt=6574 completion=64 (1.09s)


  턴3: prompt=7204 completion=64 (1.49s)

=== GPT 5.4 mini (openai/gpt-5.4-mini) — 3턴 ===


  턴1: prompt=5943 completion=35 (1.01s)


...

  턴2: prompt=11101 completion=44 (2.13s)


  턴3: prompt=12154 completion=64 (2.30s)

=== Gemini 3.6 flash (gemini/gemini-3.6-flash) — 3턴 ===


  턴1: prompt=6265 completion=60 (1.55s)


  턴2: prompt=7095 completion=60 (1.45s)


  턴3: prompt=7926 completion=60 (1.10s)

완료: openai/gpt-5.6-luna, openai/gpt-5.4-mini, anthropic/claude-opus-4-8, gemini/gemini-3.6-flash


In [6]:
# === ① usage 원본 덤프 — 정규화 없이 그대로 ===
# 여기가 이 노트북의 핵심입니다. 같은 LiteLLM SDK 인데 프로바이더마다 키 구성이 다릅니다.
for k, v in ok.items():
    print(f"\n{'=' * 78}\n{v['cfg']['label']}  ({k})\n{'=' * 78}")
    for r in v["rows"]:
        print(f"[턴{r['turn']}]")
        print(json.dumps(r["usage"], ensure_ascii=False, indent=2, default=str))


GPT 5.6 luna  (openai/gpt-5.6-luna)
[턴1]
{
  "completion_tokens": 64,
  "prompt_tokens": 5943,
  "total_tokens": 6007,
  "completion_tokens_details": {
    "accepted_prediction_tokens": 0,
    "audio_tokens": 0,
    "reasoning_tokens": 64,
    "rejected_prediction_tokens": 0,
    "text_tokens": null,
    "image_tokens": null,
    "video_tokens": null
  },
  "prompt_tokens_details": {
    "audio_tokens": 0,
    "cached_tokens": 5940,
    "text_tokens": null,
    "image_tokens": null,
    "video_tokens": null,
    "cache_write_tokens": 0
  }
}
[턴2]
{
  "completion_tokens": 64,
  "prompt_tokens": 6574,
  "total_tokens": 6638,
  "completion_tokens_details": {
    "accepted_prediction_tokens": 0,
    "audio_tokens": 0,
    "reasoning_tokens": 64,
    "rejected_prediction_tokens": 0,
    "text_tokens": null,
    "image_tokens": null,
    "video_tokens": null
  },
  "prompt_tokens_details": {
    "audio_tokens": 0,
    "cached_tokens": 6571,
    "text_tokens": null,
    "image_tokens": null,

In [7]:
# === ② 캐시 관련 키만 뽑아 비교 ===
# usage 전체에서 'cach' 가 들어간 키를 재귀적으로 찾아 경로와 값을 보여줍니다.
def find_cache_keys(obj, prefix=""):
    found = {}
    if isinstance(obj, dict):
        for key, val in obj.items():
            path = f"{prefix}.{key}" if prefix else key
            if "cach" in key.lower():
                if isinstance(val, dict):
                    found.update(find_cache_keys(val, path))
                else:
                    found[path] = val
            elif isinstance(val, dict):
                found.update(find_cache_keys(val, path))
    return found


all_paths = set()
per_model = {}
for k, v in ok.items():
    per_model[k] = [find_cache_keys(r["usage"]) for r in v["rows"]]
    for d in per_model[k]:
        all_paths.update(d)

print("발견된 캐시 관련 필드 경로:")
for p in sorted(all_paths):
    print(f"  · {p}")

print(f"\n{'모델':22s} | {'필드 경로':52s} | 턴별 값")
print("-" * 110)
for k, v in ok.items():
    label = v["cfg"]["label"]
    for path in sorted(all_paths):
        vals = []
        for d in per_model[k]:
            vals.append("—" if path not in d else str(d[path]))
        # 이 모델에서 한 번도 안 나온 경로는 '키 없음'으로 표시
        present = any(path in d for d in per_model[k])
        mark = "" if present else "  (키 없음)"
        print(f"{label:22s} | {path:52s} | {', '.join(vals)}{mark}")
    print()

발견된 캐시 관련 필드 경로:
  · cache_creation_input_tokens
  · cache_read_input_tokens
  · prompt_tokens_details.cache_creation_tokens
  · prompt_tokens_details.cache_write_tokens
  · prompt_tokens_details.cached_tokens

모델                     | 필드 경로                                                | 턴별 값
--------------------------------------------------------------------------------------------------------------
GPT 5.6 luna           | cache_creation_input_tokens                          | —, —, —  (키 없음)
GPT 5.6 luna           | cache_read_input_tokens                              | —, —, —  (키 없음)
GPT 5.6 luna           | prompt_tokens_details.cache_creation_tokens          | —, —, —  (키 없음)
GPT 5.6 luna           | prompt_tokens_details.cache_write_tokens             | 0, 0, 630
GPT 5.6 luna           | prompt_tokens_details.cached_tokens                  | 5940, 6571, 6571

GPT 5.4 mini           | cache_creation_input_tokens                          | —, —, —  (키 없음)
GPT 5.4 mini         

In [8]:
# === ③ 정규화 요약 — 턴별 read / write ===
# 프로바이더가 무엇을 쓰든 아래 우선순위로 하나의 read/write 값을 뽑습니다.
READ_PATHS = (
    ("prompt_tokens_details", "cached_tokens"),   # 3사 공통 (Gemini 는 미스 시 None)
    ("cache_read_input_tokens",),                 # Anthropic 최상위 (중복 제공)
)
WRITE_PATHS = (
    ("prompt_tokens_details", "cache_write_tokens"),    # OpenAI 5.6+
    ("prompt_tokens_details", "cache_creation_tokens"),  # Anthropic
    ("cache_creation_input_tokens",),                    # Anthropic 최상위 (중복 제공)
)


def dig(usage: dict, path: tuple):
    """중첩 경로를 따라가 정수만 반환. 키 부재/None/비정수는 전부 None."""
    cur = usage
    for key in path:
        if not isinstance(cur, dict) or key not in cur:
            return None
        cur = cur[key]
    return cur if isinstance(cur, int) else None


def pick(usage: dict, paths) -> int:
    """첫 번째로 값이 있는 경로를 채택. 아무데도 없으면 0."""
    for p in paths:
        v = dig(usage, p)
        if v is not None:
            return v
    return 0


print(f"{'모델':22s} | {'턴':>3} | {'prompt':>8} | {'read':>8} | {'write':>8} | {'적중률':>7}")
print("-" * 78)
summary = {}
for k, v in ok.items():
    label = v["cfg"]["label"]
    per_turn = []
    for r in v["rows"]:
        u = r["usage"]
        p = u.get("prompt_tokens") or 0
        read, write = pick(u, READ_PATHS), pick(u, WRITE_PATHS)
        rate = read / p if p else 0
        per_turn.append({"turn": r["turn"], "prompt": p, "read": read, "write": write, "rate": rate})
        print(f"{label:22s} | {r['turn']:>3} | {p:>8,} | {read:>8,} | {write:>8,} | {rate:>6.1%}")
    summary[k] = per_turn
    print()

print("※ 턴1 의 read 가 0 인 것은 정상입니다 — 캐시에 아직 아무것도 없습니다.")
print("※ write 가 계속 0 인 모델은 '쓰기 무료'(GPT 5.5 이하 / Gemini)이거나 캐시가 안 걸린 것입니다.")

모델                     |   턴 |   prompt |     read |    write |     적중률
------------------------------------------------------------------------------
GPT 5.6 luna           |   1 |    5,943 |    5,940 |        0 |  99.9%
GPT 5.6 luna           |   2 |    6,574 |    6,571 |        0 | 100.0%
GPT 5.6 luna           |   3 |    7,204 |    6,571 |      630 |  91.2%

GPT 5.4 mini           |   1 |    5,943 |    4,864 |        0 |  81.8%
GPT 5.4 mini           |   2 |    6,601 |    5,376 |        0 |  81.4%
GPT 5.4 mini           |   3 |    7,252 |    6,400 |        0 |  88.3%

Claude Opus 4.8        |   1 |   10,022 |        0 |   10,020 |   0.0%
Claude Opus 4.8        |   2 |   11,101 |   10,020 |    1,079 |  90.3%
Claude Opus 4.8        |   3 |   12,154 |   11,099 |    1,053 |  91.3%

Gemini 3.6 flash       |   1 |    6,265 |        0 |        0 |   0.0%
Gemini 3.6 flash       |   2 |    7,095 |        0 |        0 |   0.0%
Gemini 3.6 flash       |   3 |    7,926 |        0 |        0 |  

In [9]:
# === ③-b Gemini 캐시 히트 확보 — 멀티턴만으로는 샘플이 안 나온다 ===
# 위 3턴 멀티턴에서 Gemini 는 read 가 계속 0 입니다. 턴마다 프리픽스가 바뀌는데
# 암시적 캐싱은 워밍업 회차가 필요해서 짧은 멀티턴을 따라오지 못하기 때문입니다.
# 그래서 '큰 프리픽스를 그대로 반복' 하는 방식으로 히트 샘플을 따로 확보합니다.
_UNIT = (
    "The distributed order-processing service uses idempotency keys and an inbox table to guarantee "
    "that redelivered messages do not duplicate side effects. Under at-least-once delivery the consumer "
    "must be idempotent: derive the key from the message id and insert it in the same transaction. "
)
GEMINI_PREFIX = _UNIT * 260          # 약 14K 토큰 — 최소 임계(1024)보다 훨씬 크게
GEMINI_TRIES = 6                     # 실측상 4회차쯤부터 걸립니다

gemini_hit_rows = []
_msgs = [{"role": "user", "content": GEMINI_PREFIX + "\n\nSay OK."}]
print(f"gemini/gemini-3.6-flash · 동일 요청 {GEMINI_TRIES}회 반복")
for i in range(1, GEMINI_TRIES + 1):
    try:
        resp = litellm.completion(model="gemini/gemini-3.6-flash", messages=_msgs, max_tokens=8)
    except Exception as e:
        print(f"  {i}회차 ERR {type(e).__name__}: {str(e)[:160]}")
        break
    u = usage_to_dict(resp.usage)
    det = u.get("prompt_tokens_details") or {}
    p = u.get("prompt_tokens") or 0
    cached, top = det.get("cached_tokens"), u.get("cache_read_input_tokens")
    gemini_hit_rows.append({"try": i, "prompt": p, "details_cached": cached, "top_cache_read": top})
    rate = (cached or 0) / p if p else 0
    print(f"  {i}회차: prompt={p:,} | details.cached_tokens={cached!r} | "
          f"top.cache_read_input_tokens={top!r} | 적중 {rate:.1%}")

_hits = [r for r in gemini_hit_rows if r["details_cached"]]
if _hits:
    first = _hits[0]["try"]
    print(f"\n→ {first}회차부터 적중. 히트했을 때 Gemini 도 **두 군데 모두** 값을 줍니다:")
    print(f"   prompt_tokens_details.cached_tokens = {_hits[0]['details_cached']:,}")
    print(f"   최상위 cache_read_input_tokens      = {_hits[0]['top_cache_read']:,}")
    print("   → Anthropic 과 마찬가지로 **둘 다 더하면 이중 계산**입니다. 하나만 쓰세요.")
    print(f"\n   적중률이 {(_hits[0]['details_cached'] / _hits[0]['prompt']):.0%} 수준인 것도 특징입니다 —")
    print("   Gemini 암시적 캐싱은 프리픽스 전량이 아니라 일부만 잡습니다(부분 캐싱).")
else:
    print("\n⚠️ 이번엔 한 번도 적중하지 않았습니다. GEMINI_TRIES 를 늘리거나 프리픽스를 더 키워 재실행하세요.")
    print("   암시적 캐싱은 보장되지 않아 실행할 때마다 결과가 달라집니다.")

gemini/gemini-3.6-flash · 동일 요청 6회 반복


  1회차: prompt=14,306 | details.cached_tokens=8173 | top.cache_read_input_tokens=8173 | 적중 57.1%


  2회차: prompt=14,306 | details.cached_tokens=8173 | top.cache_read_input_tokens=8173 | 적중 57.1%


  3회차: prompt=14,306 | details.cached_tokens=8173 | top.cache_read_input_tokens=8173 | 적중 57.1%


  4회차: prompt=14,306 | details.cached_tokens=8173 | top.cache_read_input_tokens=8173 | 적중 57.1%


  5회차: prompt=14,306 | details.cached_tokens=8173 | top.cache_read_input_tokens=8173 | 적중 57.1%


  6회차: prompt=14,306 | details.cached_tokens=8173 | top.cache_read_input_tokens=8173 | 적중 57.1%

→ 1회차부터 적중. 히트했을 때 Gemini 도 **두 군데 모두** 값을 줍니다:
   prompt_tokens_details.cached_tokens = 8,173
   최상위 cache_read_input_tokens      = 8,173
   → Anthropic 과 마찬가지로 **둘 다 더하면 이중 계산**입니다. 하나만 쓰세요.

   적중률이 57% 수준인 것도 특징입니다 —
   Gemini 암시적 캐싱은 프리픽스 전량이 아니라 일부만 잡습니다(부분 캐싱).


## ④ 프로젝트에 넣을 추출 헬퍼 (제안)

`proj-in4u-bxg2.0-api` 의 `litellm_provider.py` 는 지금 캐시 토큰을 읽지 않습니다.
아래는 기존 `_extract_reasoning_tokens` / `_usage_int` 와 같은 스타일의 추가 헬퍼입니다.
(이 노트북에서 실측한 필드 구성을 그대로 반영했습니다.)

```python
def _extract_cache_tokens(usage) -> tuple[int | None, int | None]:
    """Pull (cache_read, cache_write) out of a LiteLLM usage object.

    LiteLLM normalises reads onto ``prompt_tokens_details.cached_tokens`` for
    every provider, but writes keep provider-specific names:

    - OpenAI GPT-5.6+ : ``prompt_tokens_details.cache_write_tokens``
    - Anthropic       : ``prompt_tokens_details.cache_creation_tokens`` and a
                        top-level ``cache_creation_input_tokens`` (duplicated)
    - GPT-5.5 and older / Gemini: no write field at all (writes are free)

    Absent values arrive as a missing key, ``None``, or ``0`` depending on the
    provider, so every lookup is guarded. Returns ``(None, None)`` when the
    response carries no usage at all.
    """
    if usage is None:
        return None, None

    details = getattr(usage, "prompt_tokens_details", None)

    def _int(obj, name: str) -> int | None:
        value = getattr(obj, name, None) if obj is not None else None
        return value if isinstance(value, int) else None

    read = _int(details, "cached_tokens")
    if read is None:
        read = _int(usage, "cache_read_input_tokens")

    write = _int(details, "cache_write_tokens")
    if write is None:
        write = _int(details, "cache_creation_tokens")
    if write is None:
        write = _int(usage, "cache_creation_input_tokens")

    return read, write
```

> ⚠️ Anthropic 은 같은 값을 최상위와 `prompt_tokens_details` 양쪽에 주므로 **하나만 채택**해야 합니다.
> 위 헬퍼가 `if ... is None` 으로 순차 폴백하는 이유입니다 — 둘을 더하면 이중 계산이 됩니다.


In [10]:
# === 헬퍼 검증 — 이번 실행 데이터에 위 로직을 그대로 적용 ===
print("각 모델 마지막 턴 기준:")
for k, v in ok.items():
    u = v["rows"][-1]["usage"]
    print(f"  {v['cfg']['label']:22s} read={pick(u, READ_PATHS):>8,}  write={pick(u, WRITE_PATHS):>8,}")

print("\n'값 없음' 이 어떤 형태로 오는지 (헬퍼가 방어해야 할 케이스):")
for k, v in ok.items():
    u = v["rows"][-1]["usage"]
    det = u.get("prompt_tokens_details") or {}
    for field in ("cached_tokens", "cache_write_tokens", "cache_creation_tokens"):
        if field not in det:
            form = "키 없음"
        elif det[field] is None:
            form = "None ⚠️ (그냥 더하면 TypeError)"
        else:
            form = f"{det[field]:,}"
        print(f"  {v['cfg']['label']:22s} prompt_tokens_details.{field:22s} → {form}")
    print()

각 모델 마지막 턴 기준:
  GPT 5.6 luna           read=   6,571  write=     630
  GPT 5.4 mini           read=   6,400  write=       0
  Claude Opus 4.8        read=  11,099  write=   1,053
  Gemini 3.6 flash       read=       0  write=       0

'값 없음' 이 어떤 형태로 오는지 (헬퍼가 방어해야 할 케이스):
  GPT 5.6 luna           prompt_tokens_details.cached_tokens          → 6,571
  GPT 5.6 luna           prompt_tokens_details.cache_write_tokens     → 630
  GPT 5.6 luna           prompt_tokens_details.cache_creation_tokens  → 키 없음

  GPT 5.4 mini           prompt_tokens_details.cached_tokens          → 6,400
  GPT 5.4 mini           prompt_tokens_details.cache_write_tokens     → 키 없음
  GPT 5.4 mini           prompt_tokens_details.cache_creation_tokens  → 키 없음

  Claude Opus 4.8        prompt_tokens_details.cached_tokens          → 11,099
  Claude Opus 4.8        prompt_tokens_details.cache_write_tokens     → 키 없음
  Claude Opus 4.8        prompt_tokens_details.cache_creation_tokens  → 1,053

  Gemini 3.6 flash       p

In [11]:
# === 결과 저장 ===
payload = {
    "litellm_version": metadata.version("litellm"),
    "openai_version": metadata.version("openai"),
    "n_turns": N_TURNS,
    "cache_field_paths_found": sorted(all_paths),
    "models": {
        k: {
            "label": v["cfg"]["label"],
            "needs_cache_control": v["cfg"]["needs_cache_control"],
            "turns": summary.get(k, []),
            "raw_usage": [r["usage"] for r in v["rows"]],
        }
        for k, v in ok.items()
    },
    "failed": [k for k, v in results.items() if not v["rows"]],
    # 멀티턴으로는 안 잡히는 Gemini 히트 샘플 (③-b 셀)
    "gemini_repeat_probe": gemini_hit_rows,
}
out = pathlib.Path("litellm_cache_fields_results.json")
out.write_text(json.dumps(payload, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
print(f"저장: {out.resolve()}")

저장: /Users/user/jinsup_space/research/notebooks/kv_cache/litellm/litellm_cache_fields_results.json


## 결론 — `proj-in4u-bxg2.0-api` 에 반영할 것

### 1. 읽기는 하나로 통일, 쓰기는 아니다
LiteLLM 은 **읽기(read)를 `prompt_tokens_details.cached_tokens` 로 정규화**해 줍니다. 3사 모두 여기로 옵니다.
하지만 **쓰기(write)는 프로바이더 이름을 그대로 둡니다** — `cache_write_tokens`(OpenAI 5.6+) vs `cache_creation_tokens`(Anthropic).
그래서 읽기만 볼 거면 한 줄이면 되지만, **쓰기까지 정확히 계측하려면 분기가 필요**합니다.

### 2. "값이 없다"가 세 가지 형태로 온다
| 형태 | 예 |
|---|---|
| 키 자체가 없음 | `gpt-5.4-mini` 의 `cache_write_tokens` |
| `None` | `gemini` 의 `cached_tokens` (캐시 미스일 때) |
| `0` | `gpt-5.6-luna` 의 웜 요청 `cache_write_tokens` |

기존 `_usage_int` 가 `isinstance(value, int)` 로 거르는 것과 같은 방어가 캐시 필드에도 필요합니다.
특히 **Gemini 의 `None` 을 그냥 더하면 `TypeError`** 가 납니다.

### 3. 최상위와 details 에 같은 값이 중복으로 온다 (Anthropic · Gemini)
Anthropic 은 `cache_creation_input_tokens`(최상위) = `prompt_tokens_details.cache_creation_tokens`,
`cache_read_input_tokens`(최상위) = `cached_tokens` 입니다.
**Gemini 도 히트했을 때는 최상위 `cache_read_input_tokens` 에 같은 값을 줍니다**(③-b 셀에서 확인).
**둘 다 더하면 이중 계산**이 되므로 하나만 쓰세요 — 위 헬퍼가 순차 폴백하는 이유입니다.

### 3-b. Gemini 는 짧은 멀티턴에서 캐시를 기대하기 어렵다
3턴 멀티턴에서 Gemini 는 `read` 가 계속 0 이었습니다. 턴마다 프리픽스가 바뀌는데
암시적 캐싱은 워밍업 회차가 필요해서 따라오지 못합니다.
③-b 셀처럼 **같은 큰 프리픽스를 4회 이상 반복**해야 비로소 잡히고, 그마저 **부분 캐싱**(전량이 아님)입니다.
에이전트 워크로드에서 Gemini 캐싱을 비용 계획에 넣을 때 **보수적으로 잡아야 하는 이유**입니다.

### 4. 원가 계산에 미치는 영향
지금 `LiteLLMProvider` 는 캐시를 안 보므로 `prompt_tokens` 전체를 정가로 계산하게 됩니다.

- **캐시 읽기가 많은 워크로드** — 읽기는 입력가의 10% 인데 100% 로 기록되어 **원가가 과대 계상**됩니다.
- **GPT 5.6 계열** — 쓰기는 입력가의 125% 인데 100% 로 기록되어 **과소 계상**됩니다.

`llm_models` 테이블에 `input_price_per_million` / `output_price_per_million` 만 있으므로,
캐시 단가를 반영하려면 **cached/write 단가 컬럼 추가**도 함께 검토해야 합니다
(LiteLLM 의 `litellm.model_cost` 에는 `cache_read_input_token_cost` / `cache_creation_input_token_cost` 가 이미 들어 있어,
`model_catalog.py` 의 `_litellm_pricing` 을 확장하면 자동으로 채울 수 있습니다).

### 5. 멀티턴에서 Anthropic 만 손이 더 간다
`cache_control` 을 **매 턴 마지막 user 블록으로 옮겨줘야** 이전 대화까지 캐시가 확장됩니다.
지금 `_msg_to_dict` / `_render_content` 는 `cache_control` 을 붙이지 않으므로,
**Anthropic 계열 모델은 멀티턴에서 캐시가 거의 안 걸리고 있을 가능성이 높습니다**
(Bedrock/Vertex 경유의 Claude 도 동일). 이건 비용에 직접 영향을 주는 지점입니다.
